# Ex-ante Probabilistic LCA — minimal workflow

A single **Python** pipeline: presampling → Monte Carlo → GSA on a parameterised Brightway2 model.

It consolidates a workflow that originally spanned three tools — **R** (presampling),
**Activity Browser** (LCA calculation), and **MATLAB** (GSA) — into one notebook.
Method, model, and case study: Blanco et al. (2024), *J. Industrial Ecology*
(see the repository README / Acknowledgments). `MC_workflow.ipynb` is the recommended,
structured version; this one keeps everything inline for a quick read of the mechanism.

## Notes baked in as guards

- **Kernel:** use a **numpy < 2** environment (`ab_new`, `ab`, `premise`). `bw2data 3.6.x`
  crashes on numpy ≥ 2 (`np.NaN` removed). Section 0 warns you.
- **Scan all parameterised databases** (here `SiTaSol_F2v1a` *and* `IEA_PVPS_2020`), not just
  the foreground — otherwise some parameters silently do nothing and get a spurious zero in the GSA.
- **Reproducibility principle:** every parameterised exchange is overwritten from the
  parameters in every iteration, so the result depends only on the sample matrix — not on
  leftover database state. On a clean database this reproduces the original parameter approach
  exactly (`0.136787, 0.188661, 0.105696, …`).

> Note: while porting the calculation we found the Activity Browser *scenario* export was not
> reproducible (different numbers on identical inputs across runs). This scripted pipeline is
> deterministic given the sample matrix — another reason to keep the whole chain in Python.

## 0. Imports, kernel check, project setup

In [ ]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import beta as beta_dist, bernoulli, norm, triang, uniform, lognorm

import bw2data as bd
import brightway2 as bw
from SALib.analyze import delta

if int(np.__version__.split('.')[0]) >= 2:
    print(f"WARNING: numpy {np.__version__}. bw2data 3.6.6 needs numpy < 2 "
          f"(LCA crashes with 'np.NaN removed'). Switch to the ab_new / ab / premise kernel.")
else:
    print(f"numpy {np.__version__} OK")

In [ ]:
bd.projects.set_current("FINAL")

pv   = bw.Database("SiTaSol_F2v1a")
iea  = bw.Database("IEA_PVPS_2020")
elec = pv.get("ba8e2884bda6251e448284b8d4e9cc15_copy1")
ipcc = bw.Method(('ILCD 2.0 2018 midpoint', 'climate change', 'climate change total'))

print("Project:", bd.projects.current)
print("FU     :", elec)
print("Method :", ipcc.name)

## 1. Presampling

Generates the 21 parameter samples and writes `PROB_X_t0.csv` (same seed as the original). Skip if you already have the CSV.

In [ ]:
np.random.seed(42)
replications = 10000

def rpert(size, min, mode, max, shape=4):
    alpha  = 1 + shape * (mode - min) / (max - min)
    beta_p = 1 + shape * (max - mode) / (max - min)
    return beta_dist.rvs(alpha, beta_p, size=size) * (max - min) + min

Eff_PV        = rpert(replications, min=0.25, mode=0.28, max=0.31, shape=4)
PR_PV         = rpert(replications, min=0.8,  mode=0.85, max=0.9,  shape=4)
LT            = norm.rvs(30, 5, size=replications)
Irrad         = uniform.rvs(1500, 500, size=replications)
RT_movpe      = rpert(replications, min=0.5,  mode=3.5, max=3.5,  shape=4)
P_movpe_tool  = rpert(replications, min=1,    mode=509, max=509)
Zeol_scrub    = triang.rvs(c=0, loc=2.55, scale=7.65-2.55, size=replications)
Cu_zeol       = rpert(replications, min=0.2,  mode=0.3, max=0.7,  shape=4)
Cu_rec        = bernoulli.rvs(0.5, size=replications)
pi_NPsynthCu  = rpert(replications, 0.5, 0.7, 0.8); bin_NPsynthCu = bernoulli.rvs(pi_NPsynthCu)
pi_NPsynthAg  = rpert(replications, 0.5, 0.7, 0.8); bin_NPsynthAg = bernoulli.rvs(pi_NPsynthAg)
pi_CuSint     = rpert(replications, min=0.1, mode=0.2, max=0.3, shape=4); bin_CuSint = bernoulli.rvs(pi_CuSint)
pi_AgSint     = uniform.rvs(0, 1, size=replications); bin_AgSint = bernoulli.rvs(pi_AgSint)
pi_FM         = beta_dist.rvs(4, 2, size=replications); bin_FM = bernoulli.rvs(pi_FM)
Elec_Siem     = lognorm.rvs(s=np.log(1.22), scale=np.exp(np.log(110)),   size=replications)
Heat_Siem     = lognorm.rvs(s=np.log(1.22), scale=np.exp(np.log(185)),   size=replications)
Elec_CZ       = lognorm.rvs(s=np.log(1.22), scale=np.exp(np.log(85.26)), size=replications)
scSi_CZ       = lognorm.rvs(s=np.log(1.22), scale=np.exp(np.log(1.07)),  size=replications)
Al_panel      = lognorm.rvs(s=np.log(1.22), scale=np.exp(np.log(2.63)),  size=replications)
Glass_panel   = lognorm.rvs(s=np.log(1.22), scale=np.exp(np.log(10.08)), size=replications)
Elec_panel    = lognorm.rvs(s=np.log(1.22), scale=np.exp(np.log(4.71)),  size=replications)

param_names = [
    'Irrad', 'Eff_PV', 'PR_PV', 'LT',
    'Elec_Siem', 'Heat_Siem', 'Elec_CZ', 'scSi_CZ',
    'RT_movpe', 'P_movpe_tool', 'Zeol_scrub', 'Cu_zeol', 'Cu_rec',
    'bin_NPsynthCu', 'bin_NPsynthAg', 'bin_CuSint', 'bin_AgSint', 'bin_FM',
    'Al_panel', 'Glass_panel', 'Elec_panel'
]
var_level = {
    'Irrad':'project','Eff_PV':'project','PR_PV':'project','LT':'project',
    'Elec_Siem':'activity','Heat_Siem':'activity','Elec_CZ':'activity','scSi_CZ':'activity',
    'RT_movpe':'project','P_movpe_tool':'project','Zeol_scrub':'project',
    'Cu_zeol':'project','Cu_rec':'project','bin_NPsynthCu':'project','bin_NPsynthAg':'project',
    'bin_CuSint':'project','bin_AgSint':'project','bin_FM':'project',
    'Al_panel':'project','Glass_panel':'project','Elec_panel':'project'
}

df_samples = pd.DataFrame({n: eval(n) for n in param_names})
df_T = df_samples.T.copy()
df_T.index.name = 'Name'
df_T.insert(0, 'Group', df_T.index.map(var_level))
df_T.to_csv('PROB_X_t0.csv')
print(f"Saved PROB_X_t0.csv  ({replications} replications, {len(param_names)} parameters)")

## 2. Load presampled X

Use `PROB_X_t0.csv` (full 10 000) or `PROB_X_t0_short.csv` (first 10, for quick verification).

In [ ]:
CSV_FILE = 'PROB_X_t0.csv'

with open(CSV_FILE, 'r') as f:
    reader = csv.reader(f)
    next(reader)
    rows = list(reader)

param_names = [row[0] for row in rows]
X = np.array([row[2:] for row in rows], dtype=float).T   # (replications, n_params)

print(f"File       : {CSV_FILE}")
print(f"Parameters : {param_names}")
print(f"X shape    : {X.shape}")

## 3. Pre-index formula exchanges in BOTH databases

Scanning `iea` as well as `pv` is what makes the 4 Si-supply-chain parameters (`Elec_Siem`, `Heat_Siem`, `Elec_CZ`, `scSi_CZ`) actually affect the result. If you only scan `pv`, those 4 parameters do nothing and their GSA sensitivity would be a spurious zero.

In [ ]:
formula_exchanges = []
for db in (pv, iea):
    for act in db:
        for exc in act.exchanges():
            if 'formula' in exc:
                formula_exchanges.append(exc)

print(f"Formula exchanges (pv + iea): {len(formula_exchanges)}")
for exc in formula_exchanges:
    print(f"  {exc['formula']}")

## 4. Verification — 5 scenarios

Reference = the reproducible clean-Python values (identical to the original `Setup.ipynb` parameter approach). These are **not** the AB scenario export, which is unreliable (see top of notebook).

In [ ]:
def run_mc(X, param_names, formula_exchanges, n_iter=None):
    """For each scenario i: evaluate every formula exchange (both DBs) from the
    parameters and run LCA. State-independent: all parameterised exchanges are
    overwritten each iteration, so prior database contamination cannot affect it."""
    if n_iter is None:
        n_iter = X.shape[0]
    scores = []
    for i in range(n_iter):
        params_i = dict(zip(param_names, X[i].tolist()))
        for exc in formula_exchanges:
            try:
                exc['amount'] = float(eval(exc['formula'], {"__builtins__": None}, params_i))
                exc.save()
            except Exception:
                pass
        lca = bw.LCA({elec: 1}, ipcc.name)
        lca.lci(); lca.lcia()
        scores.append(lca.score)
    return np.array(scores)


ref = [0.136787, 0.188661, 0.105696, 0.148844, 0.156320]   # reproducible clean-Python reference
Y_test = run_mc(X, param_names, formula_exchanges, n_iter=5)

print(f"{'Sc':>3}  {'Python':>12}  {'reference':>12}  {'Diff':>12}")
print('-' * 46)
for i, (py, rf) in enumerate(zip(Y_test, ref)):
    print(f"{i:>3}  {py:>12.6f}  {rf:>12.6f}  {py-rf:>+12.6f}")

## 5. Full MC run (10 000 iterations)

Run only after the verification matches.

In [ ]:
import time
t0 = time.time()
Y = run_mc(X, param_names, formula_exchanges)
print(f"Done in {time.time()-t0:.0f}s.  n={len(Y)}  mean={Y.mean():.4f}  std={Y.std():.4f}")

pd.DataFrame({'score': Y}).to_csv('Model_results_python.csv', index=False)
print('Saved Model_results_python.csv')

## 6. Visualise MC results

In [ ]:
# Y = pd.read_csv('Model_results_python.csv')['score'].values   # load if needed

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].hist(Y, bins=50)
axs[0].set_title('Histogram – GWI electricity production')
axs[0].set_xlabel('kg CO\u2082 eq / kWh')
axs[1].boxplot(Y)
axs[1].set_title('Boxplot – GWI electricity production')
axs[1].set_xticks([])
plt.tight_layout(); plt.show()
print(f"Mean={Y.mean():.4f}  Std={Y.std():.4f}  Min={Y.min():.4f}  Max={Y.max():.4f}")

## 7. GSA — Delta sensitivity analysis (Borgonovo)

In [ ]:
problem = {
    'num_vars': X.shape[1],
    'names': param_names,
    'bounds': list(zip(X.min(axis=0), X.max(axis=0)))
}

Si = delta.analyze(problem, X, Y)
df_gsa = Si.to_df()
print(df_gsa.sort_values('delta', ascending=False).to_string())

In [ ]:
df_delta = df_gsa[['delta']].T
plt.figure(figsize=(14, 3))
sns.heatmap(df_delta, fmt='.4f', cmap='coolwarm', annot=True, annot_kws={'size': 7},
            cbar_kws={'label': 'Delta sensitivity measure'})
plt.tight_layout()
plt.savefig('GSA_heatmap.png', dpi=150)
plt.show()
df_gsa.to_csv('GSA_results.csv')
print('Saved GSA_results.csv and GSA_heatmap.png')